In [43]:
import pandas as pd
import numpy as np

In [45]:
np.random.seed(42)

In [47]:
num_products = 10
num_days = 90

In [49]:
data = []
start_date = pd.Timestamp('2024-01-01')

In [51]:
for product in range(1, num_products + 1):
   
    if product <= 3:
        demand_mean = 35
        demand_std = 8
        stock = np.random.randint(20, 60)   # low stock → stockouts
        restock_threshold = 20
        restock_prob = 0.3

    elif product <= 6:
        demand_mean = 20
        demand_std = 6
        stock = np.random.randint(50, 100)  # balanced
        restock_threshold = 15
        restock_prob = 0.5

    else:
        demand_mean = 8
        demand_std = 3
        stock = np.random.randint(100, 180) # overstocked
        restock_threshold = 10
        restock_prob = 0.7
    
    lead_time = np.random.randint(2, 10)
    holding_cost = round(np.random.uniform(0.1, 1.0), 2)

    warehouse = np.random.choice(['A', 'B', 'C'])
    warehouse_performance = {
            'A': 0.95,
            'B': 0.85,
            'C': 0.7
    }

    for day in range(num_days):

        date = start_date + pd.Timedelta(days=day)

        seasonality_strength = max(1, 10 - product)
        seasonality = seasonality_strength * np.sin(day / 3)

        adjusted_mean = demand_mean + seasonality

        demand = max(0, int(np.random.normal(adjusted_mean, demand_std)))

        for _ in range(demand): 

            if stock > 0:
                stock -= 1
                in_stock = 1
            else:
                in_stock = 0

            actual_stock = max(stock, 0)

            fulfilled = 1 if (
                in_stock == 1 and
                np.random.rand() < warehouse_performance[warehouse]
            ) else 0
            
            data.append([
                product,
                date,
                1, 
                actual_stock,
                lead_time,
                holding_cost,
                warehouse,
                fulfilled
            ])
        if stock <= restock_threshold and np.random.rand() < restock_prob:
            stock += np.random.randint(40, 100)

In [53]:
df = pd.DataFrame(data, columns=[
    "product_id", 
    "date", 
    "demand", 
    "stock_level", 
    "lead_time", 
    "holding_cost",
    "warehouse", 
    "fulfilled"
])

In [55]:
df['order_id'] = range(1, len(df)+1)
df['fulfillment_time'] = df['lead_time'] + np.random.randint(0, 3, len(df))
df['in_stock'] = (df['stock_level'] > 0).astype(int)

In [123]:
warehouse_map = {
    product: np.random.choice(['A', 'B', 'C'])
    for product in df['product_id'].unique()
}

df['warehouse'] = df['product_id'].map(warehouse_map)

warehouse_performance = {
    'A': 0.95,
    'B': 0.85,
    'C': 0.7
}

In [125]:
df['fulfilled'] = df.apply(
    lambda row: 1 if (
        row['in_stock'] == 1 and 
        np.random.rand() < warehouse_performance[row['warehouse']]
    ) else 0,
    axis=1
)

In [57]:
df.to_csv("supply_chain_data.csv", index=False)